In [ ]:
import pandas as pd
import numpy as np

# Load the cleaned dataset from Part 1
df = pd.read_csv("cleaned_data.csv")
print(f"Loaded shape: {df.shape}")

# Cap outliers in SalePrice and Gr Liv Area (limits extreme values, keeps all rows)
def cap_iqr(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    return series.clip(lower=lower, upper=upper)

for col in ["SalePrice", "Gr Liv Area"]:
    before_max = df[col].max()
    df[col] = cap_iqr(df[col])
    print(f"{col}: capped. max before={before_max:.2f}, max after={df[col].max():.2f}")

# Regression target: predict the actual price
y_reg = df["SalePrice"]

# Classification target: is this house above or below the median price?
y_clf = (y_reg > y_reg.median()).astype(int)

print(f"\ny_reg stats:\n{y_reg.describe()}")
print(f"\ny_clf class balance:\n{y_clf.value_counts(normalize=True)}")

# Feature matrix: everything except the target and the two ID columns
drop_cols = ["SalePrice", "Order", "PID"]
X = df.drop(columns=drop_cols)
print(f"\nFeature matrix X shape: {X.shape}")
print(f"X columns: {X.columns.tolist()}")

In [ ]:
# STEP 1: Handle "None means the feature doesn't exist" columns
none_means_no_feature = ["Pool QC", "Alley", "Fence", "Misc Feature",
                          "Fireplace Qu", "Mas Vnr Type", "Garage Type",
                          "Garage Finish", "Garage Qual", "Garage Cond",
                          "Bsmt Qual", "Bsmt Cond", "Bsmt Exposure",
                          "BsmtFin Type 1", "BsmtFin Type 2"]

for col in none_means_no_feature:
    X[col] = X[col].fillna("None")

# STEP 2: Fill remaining categorical nulls with the mode
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")

remaining_nulls = X[cat_cols].isnull().sum()
print(f"\nRemaining categorical nulls before mode-fill:\n{remaining_nulls[remaining_nulls > 0]}")

for col in cat_cols:
    if X[col].isnull().sum() > 0:
        X[col] = X[col].fillna(X[col].mode()[0])

print(f"\nCategorical nulls after fill: {X[cat_cols].isnull().sum().sum()}")

# STEP 3: LABEL ENCODING for ordinal quality/condition columns (natural order)
ordinal_maps = {
    "Exter Qual":   {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "Exter Cond":   {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "Bsmt Qual":    {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "Bsmt Cond":    {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "Heating QC":   {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "Kitchen Qual": {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "Fireplace Qu": {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "Garage Qual":  {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "Garage Cond":  {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "Pool QC":      {"None": 0, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
}

for col, mapping in ordinal_maps.items():
    print(f"\n{col} raw values: {X[col].unique()}")
    X[col + "_encoded"] = X[col].map(mapping)
    unmapped = X.loc[X[col + "_encoded"].isna(), col].unique()
    print(f"{col} unmapped values (should be empty): {unmapped}")
    X = X.drop(columns=[col])

# STEP 4: ONE-HOT ENCODING for nominal columns (no natural order)
nominal_cols = ["MS Zoning", "Street", "Alley", "Lot Shape", "Land Contour",
                 "Utilities", "Lot Config", "Land Slope", "Neighborhood",
                 "Condition 1", "Condition 2", "Bldg Type", "House Style",
                 "Roof Style", "Roof Matl", "Exterior 1st", "Exterior 2nd",
                 "Mas Vnr Type", "Foundation", "Bsmt Exposure",
                 "BsmtFin Type 1", "BsmtFin Type 2", "Heating", "Central Air",
                 "Electrical", "Functional", "Garage Type", "Garage Finish",
                 "Paved Drive", "Fence", "Misc Feature", "Sale Type",
                 "Sale Condition", "MS SubClass"]

X = pd.get_dummies(X, columns=nominal_cols, drop_first=True)

print(f"\nFinal X shape after encoding: {X.shape}")
print(f"Remaining nulls in X: {X.isnull().sum().sum()}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split X and both labels together (same random_state -> same rows in same split)
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_reg_train shape: {y_reg_train.shape}, y_reg_test shape: {y_reg_test.shape}")
print(f"y_clf_train shape: {y_clf_train.shape}, y_clf_test shape: {y_clf_test.shape}")

# Fit the scaler ONLY on training features - never on test or full data
scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nX_train_scaled mean (should be ~0): {X_train_scaled.mean():.4f}")
print(f"X_train_scaled std (should be ~1): {X_train_scaled.std():.4f}")
print(f"X_test_scaled mean: {X_test_scaled.mean():.4f}")

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd

# ============================================================
# PART A: Linear Regression
# ============================================================
lin_reg = LinearRegression()
lin_reg.fit(X_train_scaled, y_reg_train)

y_pred_reg = lin_reg.predict(X_test_scaled)

mse_lin = mean_squared_error(y_reg_test, y_pred_reg)
r2_lin = r2_score(y_reg_test, y_pred_reg)

print("Linear Regression Results:")
print(f"  MSE = {mse_lin:.4f}")
print(f"  R²  = {r2_lin:.4f}")

coef_df = pd.DataFrame({
    "feature": X.columns,
    "coefficient": lin_reg.coef_
})
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coefficient", ascending=False)

print("\nTop 10 coefficients by absolute value:")
print(coef_df.head(10)[["feature", "coefficient"]].to_string(index=False))

print("\nTop 3 features by absolute coefficient value:")
print(coef_df.head(3)[["feature", "coefficient"]].to_string(index=False))

# ============================================================
# PART B: Ridge Regression (alpha=1.0) - same split, same scaling
# ============================================================
ridge_reg = Ridge(alpha=1.0)
ridge_reg.fit(X_train_scaled, y_reg_train)

y_pred_ridge = ridge_reg.predict(X_test_scaled)

mse_ridge = mean_squared_error(y_reg_test, y_pred_ridge)
r2_ridge = r2_score(y_reg_test, y_pred_ridge)

print("\n\nRidge Regression Results (alpha=1.0):")
print(f"  MSE = {mse_ridge:.4f}")
print(f"  R²  = {r2_ridge:.4f}")

print("\nComparison table:")
comparison = pd.DataFrame({
    "Model": ["Linear Regression", "Ridge (alpha=1.0)"],
    "MSE": [mse_lin, mse_ridge],
    "R2": [r2_lin, r2_ridge]
})
print(comparison.to_string(index=False))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, classification_report,
                               roc_curve, roc_auc_score, accuracy_score)
import matplotlib.pyplot as plt

# STEP 1: Check class balance
print("y_clf_train class counts:")
print(y_clf_train.value_counts())
print("\ny_clf_train class proportions:")
print(y_clf_train.value_counts(normalize=True))

minority_pct = y_clf_train.value_counts(normalize=True).min() * 100
print(f"\nMinority class: {minority_pct:.2f}% of training samples")

# STEP 2: Train Logistic Regression
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_clf_train)

# STEP 3: Predict class labels and probabilities on test set
y_pred_clf = log_reg.predict(X_test_scaled)
y_pred_proba = log_reg.predict_proba(X_test_scaled)[:, 1]

# STEP 4: Confusion matrix + classification report
cm = confusion_matrix(y_clf_test, y_pred_clf)
print("\nConfusion Matrix:")
print(cm)

acc = accuracy_score(y_clf_test, y_pred_clf)
print(f"\nAccuracy: {acc:.4f}")

print("\nClassification Report:")
print(classification_report(y_clf_test, y_pred_clf))

# STEP 5: ROC curve + AUC
fpr, tpr, thresholds = roc_curve(y_clf_test, y_pred_proba)
auc_score = roc_auc_score(y_clf_test, y_pred_proba)
print(f"\nAUC = {auc_score:.4f}")

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color="#2563eb", linewidth=2, label=f"ROC curve (AUC = {auc_score:.3f})")
plt.plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1, label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression (Above vs Below Median Price)")
plt.annotate(f"AUC = {auc_score:.3f}", xy=(0.6, 0.2), fontsize=12,
             bbox=dict(boxstyle="round", facecolor="white", edgecolor="gray"))
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("plot7_roc_curve.png", dpi=150)
plt.show()

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
import numpy as np

y_pred_proba = log_reg.predict_proba(X_test_scaled)[:, 1]

thresholds = np.arange(0.30, 0.71, 0.10)
results = []

for t in thresholds:
    y_pred_t = (y_pred_proba >= t).astype(int)
    precision = precision_score(y_clf_test, y_pred_t, zero_division=0)
    recall = recall_score(y_clf_test, y_pred_t, zero_division=0)
    f1 = f1_score(y_clf_test, y_pred_t, zero_division=0)
    results.append({"Threshold": round(t, 2), "Precision": precision, "Recall": recall, "F1": f1})

threshold_df = pd.DataFrame(results)
print("Threshold sensitivity table:")
print(threshold_df.to_string(index=False))

best_row = threshold_df.loc[threshold_df["F1"].idxmax()]
print(f"\nBest threshold by F1-score: {best_row['Threshold']} (F1 = {best_row['F1']:.4f})")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, roc_auc_score, accuracy_score
import pandas as pd

# Baseline model (C=1.0) already trained as log_reg - reuse its predictions
y_pred_baseline = log_reg.predict(X_test_scaled)
proba_baseline = log_reg.predict_proba(X_test_scaled)[:, 1]

precision_baseline = precision_score(y_clf_test, y_pred_baseline)
recall_baseline = recall_score(y_clf_test, y_pred_baseline)
auc_baseline = roc_auc_score(y_clf_test, proba_baseline)
acc_baseline = accuracy_score(y_clf_test, y_pred_baseline)  # NEW

# Strongly regularized model (C=0.01)
log_reg_strong = LogisticRegression(C=0.01, max_iter=1000, random_state=42)
log_reg_strong.fit(X_train_scaled, y_clf_train)

y_pred_strong = log_reg_strong.predict(X_test_scaled)
proba_strong = log_reg_strong.predict_proba(X_test_scaled)[:, 1]

precision_strong = precision_score(y_clf_test, y_pred_strong)
recall_strong = recall_score(y_clf_test, y_pred_strong)
auc_strong = roc_auc_score(y_clf_test, proba_strong)
acc_strong = accuracy_score(y_clf_test, y_pred_strong)  # NEW

print("Regularization comparison:")
comparison = pd.DataFrame({
    "Model": ["C=1.0 (baseline)", "C=0.01 (strong reg.)"],
    "Accuracy": [acc_baseline, acc_strong],
    "Precision": [precision_baseline, precision_strong],
    "Recall": [recall_baseline, recall_strong],
    "AUC": [auc_baseline, auc_strong]
})
print(comparison.to_string(index=False))

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score

np.random.seed(42)
n_bootstrap = 500
n_test = len(y_clf_test)

y_clf_test_arr = y_clf_test.values if hasattr(y_clf_test, "values") else np.array(y_clf_test)

auc_diffs = []

for i in range(n_bootstrap):
    indices = np.random.choice(n_test, size=n_test, replace=True)

    y_sample = y_clf_test_arr[indices]
    proba_baseline_sample = proba_baseline[indices]
    proba_strong_sample = proba_strong[indices]

    # Skip degenerate samples where only one class is present (AUC undefined)
    if len(np.unique(y_sample)) < 2:
        continue

    auc_baseline_sample = roc_auc_score(y_sample, proba_baseline_sample)
    auc_strong_sample = roc_auc_score(y_sample, proba_strong_sample)

    auc_diffs.append(auc_baseline_sample - auc_strong_sample)

auc_diffs = np.array(auc_diffs)

mean_diff = auc_diffs.mean()
ci_lower = np.percentile(auc_diffs, 2.5)
ci_upper = np.percentile(auc_diffs, 97.5)

print(f"Bootstrap iterations used: {len(auc_diffs)} (of {n_bootstrap} requested)")
print(f"Mean AUC difference (C=1.0 - C=0.01): {mean_diff:.4f}")
print(f"95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"CI excludes zero: {ci_lower > 0 or ci_upper < 0}")

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd

rf_reg = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_reg.fit(X_train_scaled, y_reg_train)

y_pred_rf = rf_reg.predict(X_test_scaled)

mse_rf = mean_squared_error(y_reg_test, y_pred_rf)
r2_rf = r2_score(y_reg_test, y_pred_rf)

print("Random Forest Regression Results:")
print(f"  MSE = {mse_rf:.4f}")
print(f"  R²  = {r2_rf:.4f}")

# Feature importance (RF's equivalent of coefficients)
importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": rf_reg.feature_importances_
}).sort_values("importance", ascending=False)

print("\nTop 10 features by importance:")
print(importance_df.head(10).to_string(index=False))

# Full comparison table (Linear, Ridge, Random Forest)
print("\nFull regression comparison:")
full_comparison = pd.DataFrame({
    "Model": ["Linear Regression", "Ridge (alpha=1.0)", "Random Forest"],
    "MSE": [mse_lin, mse_ridge, mse_rf],
    "R2": [r2_lin, r2_ridge, r2_rf]
})
print(full_comparison.to_string(index=False))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, classification_report,
                               roc_auc_score, accuracy_score)

rf_clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_clf.fit(X_train_scaled, y_clf_train)

y_pred_rf_clf = rf_clf.predict(X_test_scaled)
y_pred_rf_proba = rf_clf.predict_proba(X_test_scaled)[:, 1]

cm_rf = confusion_matrix(y_clf_test, y_pred_rf_clf)
acc_rf = accuracy_score(y_clf_test, y_pred_rf_clf)
auc_rf = roc_auc_score(y_clf_test, y_pred_rf_proba)

print("Random Forest Classification Results:")
print(f"\nConfusion Matrix:\n{cm_rf}")
print(f"\nAccuracy: {acc_rf:.4f}")
print(f"\nClassification Report:\n{classification_report(y_clf_test, y_pred_rf_clf)}")
print(f"AUC = {auc_rf:.4f}")

# Comparison against Logistic Regression baseline
print("\nFull classification comparison:")
clf_comparison = pd.DataFrame({
    "Model": ["Logistic Regression (C=1.0)", "Logistic Regression (C=0.01)", "Random Forest"],
    "Accuracy": [acc, None, acc_rf],  # acc was computed earlier for log_reg
    "AUC": [auc_baseline, auc_strong, auc_rf]
})
print(clf_comparison.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 6))
top_features = importance_df.head(10)
plt.barh(top_features["feature"], top_features["importance"], color="#16a34a")
plt.gca().invert_yaxis()  # highest importance at top
plt.title("Random Forest — Top 10 Feature Importances (SalePrice Prediction)")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig("plot8_rf_feature_importance.png", dpi=150)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

reg_comparison = pd.DataFrame({
    "Model": ["Linear Regression", "Ridge (alpha=1.0)", "Random Forest"],
    "R2": [r2_lin, r2_ridge, r2_rf]
})

plt.figure(figsize=(8, 5))
plt.bar(reg_comparison["Model"], reg_comparison["R2"], color=["#f97316", "#2563eb", "#16a34a"])
plt.title("Regression Model Comparison — R² Score")
plt.xlabel("Model")
plt.ylabel("R² Score")
plt.ylim(0.8, 1.0)  # zoom in so the differences are visible
for i, v in enumerate(reg_comparison["R2"]):
    plt.text(i, v + 0.003, f"{v:.4f}", ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig("plot9_regression_comparison.png", dpi=150)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

clf_comparison_chart = pd.DataFrame({
    "Model": ["Logistic Reg\n(C=1.0)", "Logistic Reg\n(C=0.01)", "Random Forest"],
    "Accuracy": [acc, acc_strong, acc_rf],
    "AUC": [auc_baseline, auc_strong, auc_rf]
})

x = range(len(clf_comparison_chart))
width = 0.35

plt.figure(figsize=(9, 6))
plt.bar([i - width/2 for i in x], clf_comparison_chart["Accuracy"], width, label="Accuracy", color="#7c3aed")
plt.bar([i + width/2 for i in x], clf_comparison_chart["AUC"], width, label="AUC", color="#16a34a")
plt.xticks(x, clf_comparison_chart["Model"])
plt.title("Classification Model Comparison — Accuracy & AUC")
plt.ylabel("Score")
plt.ylim(0.9, 1.0)
plt.legend()
plt.tight_layout()
plt.savefig("plot10_classification_comparison.png", dpi=150)
plt.show()